# ÖKOBAUDAT: input preparation and data checks

This notebook defines an input schema for environmental product information and checks the availability of source measurements. The repository currently contains a source reference and an empty template; environmental observations have not been imported.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "datasets/dtd/source.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside a cloned Fitwell-AI repository")
plt.rcParams.update({"figure.facecolor": "#f5f1ea", "axes.facecolor": "#f5f1ea", "font.size": 11})
print("Repository:", ROOT.name)


Repository: repo


In [2]:
folder = ROOT / "datasets/oekobaudat"
template = pd.read_csv(folder / "environmental_records_template.csv")
print("Template columns:", list(template.columns))
print("Template observations:", len(template))
input_path = folder / "environmental_records.csv"
records = pd.read_csv(input_path) if input_path.exists() else None
if records is None:
    print("No real environmental_records.csv found. No environmental graphs have been produced.")
else:
    required = set(template.columns)
    missing = required - set(records.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    print("Actual input rows:", len(records))
    print(records.head().to_string(index=False))

Template columns: ['source_record_id', 'material_name', 'reference_unit', 'lifecycle_module', 'indicator', 'value', 'unit', 'source_url']
Template observations: 0
No real environmental_records.csv found. No environmental graphs have been produced.


In [3]:
if records is not None and not records.empty:
    records["value"] = pd.to_numeric(records["value"], errors="coerce")
    missing_values = records.isna().sum()
    print("Missing values by column:")
    print(missing_values.to_string())
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_values.plot.bar(ax=ax, color="#b97449")
    ax.set_ylabel("Rows with missing values")
    ax.set_title("Quality check on supplied environmental records")
    plt.tight_layout()
    plt.show()
else:
    print("Analysis awaits a real export. Do not invent carbon or cost values.")

Analysis awaits a real export. Do not invent carbon or cost values.


## Interpretation

The template defines required fields for a verified source export. Reference units, lifecycle modules, indicators and measurement units must be retained. Comparisons require compatible units and system boundaries. XML conversion, environmental estimation and visual-recognition training are outside the scope of this notebook.